# 04 — Feature Engineering

**Input:**  `data/processed/student_stress_with_target.csv`  (39 rows × 38 cols)

**Outputs:**
- `X_train`, `X_test`, `y_train`, `y_test` saved to `data/processed/`
- A fitted scaler saved to `models/scaler.pkl`
- A list of final feature names saved to `data/processed/feature_names.txt`

**What this notebook does:**
1. Loads the dataset with the target variable
2. Drops all stress-question columns and derived columns to prevent target leakage
3. Drops `biggest_stress_reason` (free-text; reserved for future NLP)
4. Encodes categorical variables with one-hot encoding
5. Scales numeric features (for models sensitive to feature magnitude)
6. Splits into train / test sets (stratified by class)
7. Saves the processed feature matrices for ML training

In [1]:
# Step 1: Load the dataset and inspect

import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/student_stress_with_target.csv")

print("Shape:", df.shape)
print("\nAll columns:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col}")

Shape: (39, 38)

All columns:
   1. age
   2. gender
   3. university
   4. program
   5. year
   6. cgpa
   7. study_hours
   8. attendance
   9. assignment_stress
  10. exam_count
  11. sleep_hours
  12. social_media
  13. physical_activity
  14. part_time_job
  15. screen_time
  16. financial_stress
  17. social_support
  18. stress_q1
  19. stress_q2
  20. stress_q3
  21. stress_q4
  22. stress_q5
  23. stress_q6
  24. stress_q7
  25. stress_q8
  26. stress_q9
  27. stress_q10
  28. academic_life_satisfaction
  29. biggest_stress_reason
  30. thought_break
  31. career_stress
  32. academic_performance
  33. stress_q4_rev
  34. stress_q5_rev
  35. stress_q7_rev
  36. stress_q8_rev
  37. stress_score
  38. stress_level


## Step 2 — Separate Features (X) and Target (y)

Before any ML model can run, we must divide the dataset into:

- **`y` — the target variable:** what we want to predict (`stress_level`)
- **`X` — the feature matrix:** everything the model is allowed to see

### Columns to EXCLUDE from X — and why

We must exclude any column that could "leak" information about the target,
because the target was derived from those columns.

| Excluded columns | Reason |
|---|---|
| `stress_q1` … `stress_q10` | The target was computed from these 10 questions |
| `stress_q4_rev`, `stress_q5_rev`, `stress_q7_rev`, `stress_q8_rev` | Reverse-scored versions of the above |
| `stress_score` | Directly used to create `stress_level` |
| `stress_level` | This IS the target — never a feature |
| `biggest_stress_reason` | Free-text response, not numeric; reserved for future NLP analysis |

### Columns KEPT in X

All remaining columns — demographic, academic, lifestyle, and psychological
factors that are independent of the target:

`age`, `gender`, `university`, `program`, `year`, `cgpa`, `study_hours`,
`attendance`, `assignment_stress`, `exam_count`, `sleep_hours`, `social_media`,
`physical_activity`, `part_time_job`, `screen_time`, `financial_stress`,
`social_support`, `academic_life_satisfaction`, `thought_break`,
`career_stress`, `academic_performance`

These 21 variables are the candidate predictors.

In [2]:
# Step 2: Separate features (X) and target (y), dropping leaky columns.

# Columns that MUST NOT be used as features (target leakage)
leaky_cols = (
    [f"stress_q{i}" for i in range(1, 11)]     # stress_q1 ... stress_q10
    + ["stress_q4_rev", "stress_q5_rev", "stress_q7_rev", "stress_q8_rev"]
    + ["stress_score", "stress_level"]
)

# Free-text column — reserved for future NLP analysis
text_cols = ["biggest_stress_reason"]

# The target
y = df["stress_level"].copy()

# Features: everything else except the leaky + text columns
X = df.drop(columns=leaky_cols + text_cols)

print("Target (y) distribution:")
print(y.value_counts())
print()

print(f"Feature matrix shape: {X.shape}")
print(f"Number of features  : {X.shape[1]}")
print()

print("Feature columns kept:")
for i, col in enumerate(X.columns, 1):
    print(f"  {i:2d}. {col}")

Target (y) distribution:
stress_level
Low         15
Moderate    13
High        11
Name: count, dtype: int64

Feature matrix shape: (39, 21)
Number of features  : 21

Feature columns kept:
   1. age
   2. gender
   3. university
   4. program
   5. year
   6. cgpa
   7. study_hours
   8. attendance
   9. assignment_stress
  10. exam_count
  11. sleep_hours
  12. social_media
  13. physical_activity
  14. part_time_job
  15. screen_time
  16. financial_stress
  17. social_support
  18. academic_life_satisfaction
  19. thought_break
  20. career_stress
  21. academic_performance


## Step 3 — Identify Column Types

The 21 feature columns fall into different types, and each type needs a
different preprocessing strategy:

- **Numeric (continuous-ish):** can be used directly (then scaled)
- **Categorical (nominal):** must be one-hot encoded
- **Categorical (ordinal):** has a natural order (e.g., "Low", "Medium", "High")
  — could be label-encoded, but one-hot is safer for tree models
- **Free-text:** excluded (already done)

We first inspect data types and unique values to classify each column correctly.

In [3]:
# Step 3: Inspect the types and unique values of each feature.
#
# We use df.dtypes and value_counts to see which columns are:
#   - truly numeric
#   - categorical (nominal or ordinal)
#   - free-text

import pandas as pd

# Quick dtype summary
print("Data types in X:")
print(X.dtypes)
print()

# Show unique values per feature (first 10 unique values max)
print("Unique values per feature:")
for col in X.columns:
    uniques = X[col].unique()
    n_unique = len(uniques)
    sample = list(uniques)[:10]
    print(f"\n{col}  ({n_unique} unique)")
    print(f"  {sample}")

Data types in X:
age                             str
gender                          str
university                      str
program                         str
year                            str
cgpa                            str
study_hours                     str
attendance                      str
assignment_stress             int64
exam_count                      str
sleep_hours                     str
social_media                    str
physical_activity               str
part_time_job                   str
screen_time                     str
financial_stress              int64
social_support                int64
academic_life_satisfaction    int64
thought_break                   str
career_stress                 int64
academic_performance          int64
dtype: object

Unique values per feature:

age  (4 unique)
  ['20-22', 'Above 25', '23-25', '17-19']

gender  (3 unique)
  ['Female', 'Male', 'Prefer not to say']

university  (8 unique)
  ['Tribhuvan University', 'Other', 'Kat

In [4]:
# Inspect unique values for each ordinal text column.
# We need the exact strings (and their natural order) to map them to numbers.

ordinal_cols = [
    "age", "year", "cgpa", "study_hours", "attendance",
    "exam_count", "sleep_hours", "social_media",
    "physical_activity", "screen_time", "thought_break"
]

for col in ordinal_cols:
    print(f"\n=== {col} ===")
    print(X[col].value_counts())


=== age ===
age
20-22       14
23-25       13
Above 25     9
17-19        3
Name: count, dtype: int64

=== year ===
year
4th Year    18
3rd Year     9
2nd Year     7
5th Year     4
1st Year     1
Name: count, dtype: int64

=== cgpa ===
cgpa
2.5–2.99             12
3.0–3.49             11
3.5–4.0              10
Not Available Yet     5
2.0–2.49              1
Name: count, dtype: int64

=== study_hours ===
study_hours
1–2                  13
3–4                  12
Less than 1           6
5–6                   5
More than 6 hours     3
Name: count, dtype: int64

=== attendance ===
attendance
60–70%       11
71–80%       10
81–90%        9
Above 90%     6
Below 60%     3
Name: count, dtype: int64

=== exam_count ===
exam_count
3-4            15
1-2             8
5-6             8
More than 6     8
Name: count, dtype: int64

=== sleep_hours ===
sleep_hours
5–6            18
7–8            14
More than 8     4
Less than 5     3
Name: count, dtype: int64

=== social_media ===
social_media
3

In [5]:
# Compact view: for each ordinal column, show all unique values in
# their natural order (sorted), not by frequency. This fits on screen.

ordinal_cols = [
    "age", "year", "cgpa", "study_hours", "attendance",
    "exam_count", "sleep_hours", "social_media",
    "physical_activity", "screen_time", "thought_break"
]

for col in ordinal_cols:
    uniques = sorted(X[col].astype(str).unique())
    print(f"{col:22s} -> {uniques}")

age                    -> ['17-19', '20-22', '23-25', 'Above 25']
year                   -> ['1st Year', '2nd Year', '3rd Year', '4th Year', '5th Year']
cgpa                   -> ['2.0–2.49', '2.5–2.99', '3.0–3.49', '3.5–4.0', 'Not Available Yet']
study_hours            -> ['1–2', '3–4', '5–6', 'Less than 1', 'More than 6 hours']
attendance             -> ['60–70%', '71–80%', '81–90%', 'Above 90%', 'Below 60%']
exam_count             -> ['1-2', '3-4', '5-6', 'More than 6']
sleep_hours            -> ['5–6', '7–8', 'Less than 5', 'More than 8']
social_media           -> ['1–2 hours', '3–4 hours', '5–6 hours', 'Less than 1 hour', 'More than 6 hours']
physical_activity      -> ['1–2 days/week', '3–4 days/week', '5–6 days/week', 'Daily', 'Never']
screen_time            -> ['3–5 hours', '6–8 hours', 'Less than 3 hours', 'More than 8 hours']
thought_break          -> ['Never', 'Often', 'Rarely', 'Sometimes', 'Very Often']


## Step 4 — Ordinal Encoding of Ordered Categorical Features

Eleven of the 21 features are ordinal — they have a natural order
(e.g., "Never" < "Rarely" < "Sometimes" < "Often").

Ordinal encoding maps each category to an integer that reflects this
order. This preserves the ordering information, unlike one-hot encoding
which treats all categories as equally distant.

The mappings below are based on the exact unique values found in the
dataset (inspected in Step 3).

In [6]:
# Step 4: Ordinal encoding.
#
# Each ordinal feature is mapped to integers that reflect the natural order
# (lowest category → smallest integer).
#
# IMPORTANT: The strings must match the dataset exactly, including dash type.
# En-dash "–" and hyphen "-" are different characters.

ordinal_mappings = {
    "age": {
        "17-19": 1,
        "20-22": 2,
        "23-25": 3,
        "Above 25": 4,
    },
    "year": {
        "1st Year": 1,
        "2nd Year": 2,
        "3rd Year": 3,
        "4th Year": 4,
        "5th Year": 5,
    },
    "cgpa": {
        "Not Available Yet": 0,   # treat unknown as lowest
        "2.0–2.49": 1,
        "2.5–2.99": 2,
        "3.0–3.49": 3,
        "3.5–4.0": 4,
    },
    "study_hours": {
        "Less than 1": 1,
        "1–2": 2,
        "3–4": 3,
        "5–6": 4,
        "More than 6 hours": 5,
    },
    "attendance": {
        "Below 60%": 1,
        "60–70%": 2,
        "71–80%": 3,
        "81–90%": 4,
        "Above 90%": 5,
    },
    "exam_count": {
        "1-2": 1,
        "3-4": 2,
        "5-6": 3,
        "More than 6": 4,
    },
    "sleep_hours": {
        "Less than 5": 1,
        "5–6": 2,
        "7–8": 3,
        "More than 8": 4,
    },
    "social_media": {
        "Less than 1 hour": 1,
        "1–2 hours": 2,
        "3–4 hours": 3,
        "5–6 hours": 4,
        "More than 6 hours": 5,
    },
    "physical_activity": {
        "Never": 0,
        "1–2 days/week": 1,
        "3–4 days/week": 2,
        "5–6 days/week": 3,
        "Daily": 4,
    },
    "screen_time": {
        "Less than 3 hours": 1,
        "3–5 hours": 2,
        "6–8 hours": 3,
        "More than 8 hours": 4,
    },
    "thought_break": {
        "Never": 1,
        "Rarely": 2,
        "Sometimes": 3,
        "Often": 4,
        "Very Often": 5,
    },
}

# Apply the mappings. We use .map() and check for any unmapped values.
X_encoded = X.copy()

for col, mapping in ordinal_mappings.items():
    X_encoded[col] = X_encoded[col].map(mapping)
    
    # Check for any values that failed to map
    n_missing = X_encoded[col].isna().sum()
    if n_missing > 0:
        print(f"WARNING: {col} has {n_missing} unmapped values")
        print(f"  Unmapped: {X[col][X_encoded[col].isna()].unique()}")
    else:
        print(f"OK: {col} encoded")

print()
print("Shape after ordinal encoding:", X_encoded.shape)
print()
print("Sample rows:")
print(X_encoded.head())

OK: age encoded
OK: year encoded
OK: cgpa encoded
OK: study_hours encoded
OK: attendance encoded
OK: exam_count encoded
OK: sleep_hours encoded
OK: social_media encoded
OK: physical_activity encoded
OK: screen_time encoded
OK: thought_break encoded

Shape after ordinal encoding: (39, 21)

Sample rows:
   age  gender            university      program  year  cgpa  study_hours  \
0    2  Female  Tribhuvan University          BCA     4     0            1   
1    4    Male                 Other  BSc Nursing     3     3            3   
2    3    Male  Tribhuvan University    BE/B.Tech     4     4            2   
3    3    Male  Tribhuvan University          BCA     5     3            2   
4    4    Male  Tribhuvan University     BSc CSIT     4     0            2   

   attendance  assignment_stress  exam_count  ...  social_media  \
0           2                  2           1  ...             4   
1           5                  3           3  ...             5   
2           5              

## Step 5 — One-Hot Encoding of Nominal Features

Four features are **nominal** — their categories have no natural order:

- `gender`
- `university`
- `program`
- `part_time_job`

One-hot encoding creates a separate binary (0/1) column for each category.
For example, `gender` becomes:

| gender | gender_Female | gender_Male | gender_Other |
|--------|---------------|-------------|--------------|
| Female | 1             | 0           | 0            |
| Male   | 0             | 1           | 0            |
| Other  | 0             | 0           | 1            |

This avoids the false assumption that "Female < Male < Other" (which
label encoding would introduce).

We drop one column per feature (via `drop_first=True`) to avoid perfect
multicollinearity between the dummy columns.

In [7]:
# Step 5: One-hot encode the nominal columns.
#
# drop_first=True drops one column per feature to avoid perfect collinearity.
# pandas prefix= keeps track of which feature each dummy came from.

nominal_cols = ["gender", "university", "program", "part_time_job"]

X_final = pd.get_dummies(
    X_encoded,
    columns=nominal_cols,
    drop_first=True,
    dtype=int,   # ensure 0/1 rather than True/False
)

print("Shape after one-hot encoding:", X_final.shape)
print()
print("All feature columns now:")
for i, col in enumerate(X_final.columns, 1):
    print(f"  {i:2d}. {col}")
print()
print("Data types:")
print(X_final.dtypes)

Shape after one-hot encoding: (39, 37)

All feature columns now:
   1. age
   2. year
   3. cgpa
   4. study_hours
   5. attendance
   6. assignment_stress
   7. exam_count
   8. sleep_hours
   9. social_media
  10. physical_activity
  11. screen_time
  12. financial_stress
  13. social_support
  14. academic_life_satisfaction
  15. thought_break
  16. career_stress
  17. academic_performance
  18. gender_Male
  19. gender_Prefer not to say
  20. university_Kathmandu University
  21. university_Other
  22. university_Pokhara University
  23. university_Purbanchal University
  24. university_Rajarshi Janak University
  25. university_Sudurpaschim University
  26. university_Tribhuvan University
  27. program_BBS
  28. program_BCA
  29. program_BDS
  30. program_BE/B.Tech
  31. program_BHM
  32. program_BIM
  33. program_BSc CSIT
  34. program_BSc Nursing
  35. program_Law(LLB)
  36. program_MBBS
  37. part_time_job_Yes

Data types:
age                                     int64
year     

## Step 6 — Save the Engineered Feature Matrix

The final feature matrix `X_final` (39 × 37) and the target vector `y`
are saved to disk so that the ML training notebook can load them directly.

Saving as Parquet (efficient binary format) preserves column dtypes and
is faster than CSV for numerical data.

In [8]:
# Step 6: Save feature matrix and target to disk.

import os

os.makedirs("../data/processed", exist_ok=True)

# Save features and target as CSVs (they're small, no need for Parquet yet)
X_final.to_csv("../data/processed/X_features.csv", index=False)
y.to_csv("../data/processed/y_target.csv", index=False)

print("Saved:")
print("  ../data/processed/X_features.csv   shape:", X_final.shape)
print("  ../data/processed/y_target.csv     shape:", y.shape)
print()
print("Target distribution:")
print(y.value_counts())

Saved:
  ../data/processed/X_features.csv   shape: (39, 37)
  ../data/processed/y_target.csv     shape: (39,)

Target distribution:
stress_level
Low         15
Moderate    13
High        11
Name: count, dtype: int64
